IPL POINT AUTOMATION

In [31]:
import requests
from bs4 import BeautifulSoup

url = "https://crex.com/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

links = soup.find_all("a", href=True)

match_links = []

for link in links:
    if "/scoreboard/" in link["href"]:
        match_links.append("https://crex.com" + link["href"])

print(match_links[:5])
match_links = list(set(match_links))
print(match_links)

['https://crex.com/scoreboard/10BK/2AR/2nd-ODI/U/W/ban-vs-pak-2nd-odi-pakistan-tour-of-bangladesh-2026/live', 'https://crex.com/scoreboard/10OP/2EX/13th-Match/SW/62/fais-vs-psw-13th-match-pakistan-national-t20-2026/live', 'https://crex.com/scoreboard/10OQ/2EX/14th-Match/TP/61/bhwr-vs-lahw-14th-match-pakistan-national-t20-2026/info', 'https://crex.com/scoreboard/10OO/2EX/12th-Match/68/TG/karb-vs-mul-12th-match-pakistan-national-t20-2026/scorecard', 'https://crex.com/scoreboard/10ON/2EX/11th-Match/TM/63/lahb-vs-sial-11th-match-pakistan-national-t20-2026/scorecard']
['https://crex.com/scoreboard/10SO/2F1/2nd-Match/M9/MA/ing-vs-rrp-2nd-match-legends-league-cricket-2026/scorecard', 'https://crex.com/scoreboard/10OO/2EX/12th-Match/68/TG/karb-vs-mul-12th-match-pakistan-national-t20-2026/scorecard', 'https://crex.com/scoreboard/10RA/2F2/1st-Semi-Final/1EQ/1EM/neh-vs-nod-1st-semi-final-tillo-t20-cup-2026/live', 'https://crex.com/scoreboard/W5P/1X9/37th-Match/3N/NX/qf-w-vs-vic-w-37th-match-women

In [32]:
match_url = None

for link in match_links:
    if "ban-vs-pak" in link.lower():
        match_url = link
        break

if match_url:
    scorecard_link = match_url.replace("/live", "/scorecard")
    print(scorecard_link)
else:
    print("Match not found")

https://crex.com/scoreboard/10BK/2AR/2nd-ODI/U/W/ban-vs-pak-2nd-odi-pakistan-tour-of-bangladesh-2026/scorecard


In [33]:
response = requests.get(match_url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

PAK 266-8 (45.5) (Shaheen Afridi 3(9), Faheem Ashraf 9(12)) vs Bangladesh 2nd ODI live | Pakistan tour of bangladesh 2026 - CREX


In [34]:
# Step 4 → scrape scorecard
response = requests.get(scorecard_link, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

In [35]:
tables = soup.find_all("table")

print("Total tables found:", len(tables))

Total tables found: 3


In [36]:
for i, table in enumerate(tables):
    print("\n================")
    print("Table Index:", i)
    print("================")
    print(table.text[:800])


Table Index: 0
BatterRB4s6sSRSahibzada Farhan c Hridoy b Ahmed 31462067.39Maaz Sadaqat c Das b Miraz 754665163.04Shamyl Hussain c Rahman b Rana 6220027.27Mohammad Rizwan c Hossain b Miraz 44595074.58Salman Ali Agha run out Miraz  646272103.23Hussain Talat b Hossain  9131069.23Abdul Samad run out (Hossain / Miraz)  11701157.14Faheem Ashraf Batting 9121075.00Shaheen Afridi(C) c Hossain b Hossain 390033.33

Table Index: 1
BowlerOMRWERTaskin Ahmed8.005917.38Mustafizur Rahman8.004605.75Nahid Rana10.005915.90Mehidy Hasan Miraz10.023423.40Rishad Hossain8.505526.23Afif Hossain1.00808.00

Table Index: 2
BatsmanScoreOversMaaz Sadaqat103-113.0Sahibzada Farhan121-217.4Shamyl Hussain122-319.3Salman Ali Agha231-438.4Mohammad Rizwan231-539.0Hussain Talat252-642.1Abdul Samad254-742.3Shaheen Afridi(C)266-845.5


In [37]:
import pandas as pd

batting_rows = tables[0].find_all("tr")

batting_data = []

for row in batting_rows[1:]:
    cols = [c.text.strip() for c in row.find_all("td")]
    if len(cols) > 0:
        batting_data.append(cols)

batting_df = pd.DataFrame(batting_data)

batting_df.head()

,0,1,2,3,4,5
0,Sahibzada Farhan c Hridoy b Ahmed,31,46,2,0,67.39
1,Maaz Sadaqat c Das b Miraz,75,46,6,5,163.04
2,Shamyl Hussain c Rahman b Rana,6,22,0,0,27.27
3,Mohammad Rizwan c Hossain b Miraz,44,59,5,0,74.58
4,Salman Ali Agha run out Miraz,64,62,7,2,103.23


In [38]:
batting_df.columns = [
    "player",
    "runs",
    "balls",
    "fours",
    "sixes",
    "sr"
]

batting_df

,player,runs,balls,fours,sixes,sr
0,Sahibzada Farhan c Hridoy b Ahmed,31,46,2,0,67.39
1,Maaz Sadaqat c Das b Miraz,75,46,6,5,163.04
2,Shamyl Hussain c Rahman b Rana,6,22,0,0,27.27
3,Mohammad Rizwan c Hossain b Miraz,44,59,5,0,74.58
4,Salman Ali Agha run out Miraz,64,62,7,2,103.23
5,Hussain Talat b Hossain,9,13,1,0,69.23
6,Abdul Samad run out (Hossain / Miraz),11,7,0,1,157.14
7,Faheem Ashraf Batting,9,12,1,0,75.00
8,Shaheen Afridi(C) c Hossain b Hossain,3,9,0,0,33.33


In [39]:
batting_df["runs"] = pd.to_numeric(batting_df["runs"], errors="coerce")
batting_df["balls"] = pd.to_numeric(batting_df["balls"], errors="coerce")
batting_df["fours"] = pd.to_numeric(batting_df["fours"], errors="coerce")
batting_df["sixes"] = pd.to_numeric(batting_df["sixes"], errors="coerce")
batting_df["sr"] = pd.to_numeric(batting_df["sr"], errors="coerce")

batting_df

,player,runs,balls,fours,sixes,sr
0,Sahibzada Farhan c Hridoy b Ahmed,31,46,2,0,67.39
1,Maaz Sadaqat c Das b Miraz,75,46,6,5,163.04
2,Shamyl Hussain c Rahman b Rana,6,22,0,0,27.27
3,Mohammad Rizwan c Hossain b Miraz,44,59,5,0,74.58
4,Salman Ali Agha run out Miraz,64,62,7,2,103.23
5,Hussain Talat b Hossain,9,13,1,0,69.23
6,Abdul Samad run out (Hossain / Miraz),11,7,0,1,157.14
7,Faheem Ashraf Batting,9,12,1,0,75.00
8,Shaheen Afridi(C) c Hossain b Hossain,3,9,0,0,33.33


In [40]:
import numpy as np

# Playing XI
batting_df["playing_xi_points"] = 4

# Base run points
batting_df["run_points"] = batting_df["runs"]

# Boundary bonus (updated rule)
batting_df["boundary_points"] = (
    batting_df["fours"] * 4
    + batting_df["sixes"] * 6
)

# Milestone bonus
batting_df["milestone_points"] = np.select(
    [
        batting_df["runs"] >= 100,
        batting_df["runs"] >= 75,
        batting_df["runs"] >= 50,
        batting_df["runs"] >= 30
    ],
    [
        16,
        12,
        8,
        4
    ],
    default=0
)

# Not-out detection logic (UPDATED)
batting_df["not_out"] = batting_df["player"].str.lower().str.contains(
    "not out|batting",
    regex=True
)

# Duck penalty (only if OUT)
batting_df["duck_penalty"] = np.where(
    (batting_df["runs"] == 0) &
    (batting_df["balls"] > 0) &
    (batting_df["not_out"] == False),
    -2,
    0
)

# Strike rate numeric
batting_df["sr"] = pd.to_numeric(batting_df["sr"], errors="coerce")

# Strike rate bonus (balls >=10 only)
batting_df["sr_points"] = np.select(
    [
        (batting_df["balls"] >= 10) & (batting_df["sr"] >= 200),
        (batting_df["balls"] >= 10) & (batting_df["sr"] >= 170),
        (batting_df["balls"] >= 10) & (batting_df["sr"] >= 150),
        (batting_df["balls"] >= 10) & (batting_df["sr"] >= 130),
        (batting_df["balls"] >= 10) & (batting_df["sr"] < 50),
        (batting_df["balls"] >= 10) & (batting_df["sr"] < 60),
        (batting_df["balls"] >= 10) & (batting_df["sr"] < 70)
    ],
    [
        8,
        6,
        4,
        2,
        -6,
        -4,
        -2
    ],
    default=0
)

# Final batting fantasy points
batting_df["batting_points"] = (
    batting_df["playing_xi_points"]
    + batting_df["run_points"]
    + batting_df["boundary_points"]
    + batting_df["milestone_points"]
    + batting_df["sr_points"]
    + batting_df["duck_penalty"]
)

batting_df.sort_values("batting_points", ascending=False)

,player,runs,balls,fours,sixes,sr,playing_xi_points,run_points,boundary_points,milestone_points,not_out,duck_penalty,sr_points,batting_points
1,Maaz Sadaqat c Das b Miraz,75,46,6,5,163.04,4,75,54,12,False,0,4,149
4,Salman Ali Agha run out Miraz,64,62,7,2,103.23,4,64,40,8,False,0,0,116
3,Mohammad Rizwan c Hossain b Miraz,44,59,5,0,74.58,4,44,20,4,False,0,0,72
0,Sahibzada Farhan c Hridoy b Ahmed,31,46,2,0,67.39,4,31,8,4,False,0,-2,45
6,Abdul Samad run out (Hossain / Miraz),11,7,0,1,157.14,4,11,6,0,False,0,0,21
7,Faheem Ashraf Batting,9,12,1,0,75.00,4,9,4,0,True,0,0,17
5,Hussain Talat b Hossain,9,13,1,0,69.23,4,9,4,0,False,0,-2,15
8,Shaheen Afridi(C) c Hossain b Hossain,3,9,0,0,33.33,4,3,0,0,False,0,0,7
2,Shamyl Hussain c Rahman b Rana,6,22,0,0,27.27,4,6,0,0,False,0,-6,4


In [47]:
bowling_rows = tables[1].find_all("tr")

bowling_data = []

for row in bowling_rows[1:]:
    cols = [c.text.strip() for c in row.find_all("td")]
    if len(cols) > 0:
        bowling_data.append(cols)

import pandas as pd
bowling_df = pd.DataFrame(bowling_data)

bowling_df.head()

,0,1,2,3,4,5
0,Taskin Ahmed,8.0,0,59,1,7.38
1,Mustafizur Rahman,8.0,0,46,0,5.75
2,Nahid Rana,10.0,0,59,1,5.90
3,Mehidy Hasan Miraz,10.0,2,34,2,3.40
4,Rishad Hossain,8.5,0,55,2,6.23


In [49]:
bowling_df.columns = [
    "player",
    "overs",
    "maidens",
    "runs_conceded",
    "wickets",
    "econ"
]

bowling_df["overs"] = pd.to_numeric(bowling_df["overs"], errors="coerce")
bowling_df["maidens"] = pd.to_numeric(bowling_df["maidens"], errors="coerce")
bowling_df["runs_conceded"] = pd.to_numeric(bowling_df["runs_conceded"], errors="coerce")
bowling_df["wickets"] = pd.to_numeric(bowling_df["wickets"], errors="coerce")
bowling_df["econ"] = pd.to_numeric(bowling_df["econ"], errors="coerce")

bowling_df

,player,overs,maidens,runs_conceded,wickets,econ
0,Taskin Ahmed,8.0,0,59,1,7.38
1,Mustafizur Rahman,8.0,0,46,0,5.75
2,Nahid Rana,10.0,0,59,1,5.90
3,Mehidy Hasan Miraz,10.0,2,34,2,3.40
4,Rishad Hossain,8.5,0,55,2,6.23
5,Afif Hossain,1.0,0,8,0,8.00


In [50]:
lbw_bowled_bonus = {}

for idx, row in batting_df.iterrows():
    player_text = row["player"].lower()

    if " b " in player_text or "lbw" in player_text:
        words = player_text.split(" b ")
        if len(words) > 1:
            bowler = words[-1].strip()

            if bowler in lbw_bowled_bonus:
                lbw_bowled_bonus[bowler] += 1
            else:
                lbw_bowled_bonus[bowler] = 1

lbw_bowled_bonus

{'ahmed': 1, 'miraz': 2, 'rana': 1, 'hossain': 2}

In [51]:
import numpy as np

# Playing XI
bowling_df["playing_xi_points"] = 4

# Wicket points (UPDATED)
bowling_df["wicket_points"] = bowling_df["wickets"] * 30

# Maiden bonus
bowling_df["maiden_points"] = bowling_df["maidens"] * 12

# LBW/Bowled bonus
bowling_df["lbw_bowled_bonus"] = bowling_df["player"].str.lower().map(
    lbw_bowled_bonus
).fillna(0) * 8

# Wicket haul bonus
bowling_df["haul_bonus"] = np.select(
    [
        bowling_df["wickets"] >= 5,
        bowling_df["wickets"] >= 4,
        bowling_df["wickets"] >= 3
    ],
    [
        16,
        8,
        4
    ],
    default=0
)

# Economy bonus (min 2 overs)
bowling_df["econ_points"] = np.select(
    [
        (bowling_df["overs"] >= 2) & (bowling_df["econ"] < 5),
        (bowling_df["overs"] >= 2) & (bowling_df["econ"] < 6),
        (bowling_df["overs"] >= 2) & (bowling_df["econ"] < 7),
        (bowling_df["overs"] >= 2) & (bowling_df["econ"] >= 12),
        (bowling_df["overs"] >= 2) & (bowling_df["econ"] >= 11),
        (bowling_df["overs"] >= 2) & (bowling_df["econ"] >= 10)
    ],
    [
        6,
        4,
        2,
        -6,
        -4,
        -2
    ],
    default=0
)

# Total bowling points
bowling_df["bowling_points"] = (
    bowling_df["playing_xi_points"]
    + bowling_df["wicket_points"]
    + bowling_df["maiden_points"]
    + bowling_df["lbw_bowled_bonus"]
    + bowling_df["haul_bonus"]
    + bowling_df["econ_points"]
)

bowling_df.sort_values("bowling_points", ascending=False)

,player,overs,maidens,runs_conceded,wickets,econ,playing_xi_points,wicket_points,maiden_points,lbw_bowled_bonus,haul_bonus,econ_points,bowling_points
3,Mehidy Hasan Miraz,10.0,2,34,2,3.40,4,60,24,0.0,0,6,94.0
4,Rishad Hossain,8.5,0,55,2,6.23,4,60,0,0.0,0,2,66.0
2,Nahid Rana,10.0,0,59,1,5.90,4,30,0,0.0,0,4,38.0
0,Taskin Ahmed,8.0,0,59,1,7.38,4,30,0,0.0,0,0,34.0
1,Mustafizur Rahman,8.0,0,46,0,5.75,4,0,0,0.0,0,4,8.0
5,Afif Hossain,1.0,0,8,0,8.00,4,0,0,0.0,0,0,4.0
